# [GLOBE] Coimbatore Climate Data — Interactive Folium Explorer
**PCM Selection ML Model | data_3 pipeline**

This notebook provides interactive geospatial and temporal visualisations of the Coimbatore ERA5 climate dataset.

### What's inside:
- [PIN] **Map 1** — City location marker with full climate metadata popup
- [SUN] **Map 2** — Hourly GHI choropleth heatmap (day-of-year × hour)
- [TEMP] **Map 3** — Monthly temperature & humidity bubble map
- [WIND] **Map 4** — Wind rose / direction scatter on map
- [CHART] **Map 5** — RRTDHS monthly solar resource index marker
- [MAP] **Map 6** — Full feature cluster map (marker cluster per season)
- [GRAPH] **Bonus** — Plotly interactive time series embedded in the notebook


In [1]:
# ─── STEP 0: Install & Import ───────────────────────────────────────────────
# Install required packages
import subprocess
import sys

packages = ['folium', 'plotly', 'branca']
for pkg in packages:
    try:
        __import__(pkg)
        print(f'[OK] {pkg} already installed')
    except ImportError:
        print(f'[INSTALL] Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    from google.colab import files
    IS_COLAB = True
    print('[INFO] Colab environment detected')
except ImportError:
    IS_COLAB = False
    print('[INFO] Local environment detected — will load from local file path')


ModuleNotFoundError: No module named 'google'

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster, TimestampedGeoJson
from folium import plugins
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import branca.colormap as cm
import warnings
warnings.filterwarnings('ignore')

print('[OK] All imports successful')


✅ All imports successful


## [FOLDER] Load Data
Upload `climate_coimbatore.csv` from your `data_3/data/processed/` folder using the cell below, **OR** mount Google Drive if the file is there.


In [3]:
# ─── LOCAL: Load from file path ──
# For LOCAL environments, specify the CSV path directly:
csv_path = 'data/processed/climate_coimbatore.csv'  # Adjust to your local path
df = pd.read_csv("PCM-Selection-ML-model\data_3\data\processed\climate_coimbatore.csv")
print(f'Loaded: {csv_path}  |  Shape: {df.shape}')

# Uncomment below ONLY if running in Google Colab:

# from google.colab import files# print(f'Loaded: {fname}  |  Shape: {df.shape}')

# uploaded = files.upload()   # upload climate_coimbatore.csv# df = pd.read_csv(io.BytesIO(uploaded[fname]))

# import io# fname = list(uploaded.keys())[0]

NameError: name 'pd' is not defined

In [ ]:
# ── Option B: Mount Google Drive (comment out Option A if using this) ──
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/path/to/climate_coimbatore.csv')
# print(f'Shape: {df.shape}')

In [ ]:
# ─── STEP 1: Basic preprocessing ────────────────────────────────────────────
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# Create W_spd if it doesn't exist (from Wind_speed_ms or default to 0)
if 'W_spd' not in df.columns:
    if 'Wind_speed_ms' in df.columns:
        df['W_spd'] = df['Wind_speed_ms']
        print('[INFO] Created W_spd from Wind_speed_ms')
    else:
        df['W_spd'] = 0.0
        print('[INFO] W_spd column not found; created with zeros')

# Create W_dir if it doesn't exist (from Wind_dir_deg or default to 0)
if 'W_dir' not in df.columns:
    if 'Wind_dir_deg' in df.columns:
        df['W_dir'] = df['Wind_dir_deg']
        print('[INFO] Created W_dir from Wind_dir_deg')
    else:
        df['W_dir'] = 0.0
        print('[INFO] W_dir column not found; created with zeros')

# Core coordinates
LAT = df['lat'].iloc[0] if 'lat' in df.columns else 11.0
LON = df['lon'].iloc[0] if 'lon' in df.columns else 77.0
CITY = 'Coimbatore'
ALT = df['altitude_m'].iloc[0] if 'altitude_m' in df.columns else 411

# Time components (compute if missing)
df['hour']  = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
df['DOY']   = df['timestamp'].dt.dayofyear
df['date']  = df['timestamp'].dt.date

# Season mapping
def get_season(m):
    if m in [12, 1, 2]:     return 'Winter'
    elif m in [3, 4, 5]:    return 'Pre-Monsoon'
    elif m in [6, 7, 8, 9]: return 'Monsoon'
    else:                   return 'Post-Monsoon'

df['season_label'] = df['month'].apply(get_season)

# Month names
MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['month_name'] = df['month'].map(MONTH_NAMES)

print('Timestamp range:', df['timestamp'].min(), '→', df['timestamp'].max())
print('Columns available:', list(df.columns))
df.head(3)


Timestamp range: 2024-01-01 00:00:00 → 2024-12-31 23:00:00
Columns available: ['timestamp', 'avg_sdirswrf', 'T_amb', 'T_dew', 'RHum', 'W_dir', 'GHI', 'LW_down', 'cloud_cover', 'precipitation', 'P_atm', 'hour', 'month', 'DOY', 'year', 'season', 'season_code', 'SZA', 'solar_azimuth', 'ETR', 'GHI_clearsky', 'CSI', 'RRTDHS', 'sunrise_hour', 'sunset_hour', 'city', 'lat', 'lon', 'altitude_m', 'climate_zone', 'T_set', 'high_solar_resource', 'date', 'season_label', 'month_name']


,timestamp,avg_sdirswrf,T_amb,T_dew,RHum,W_dir,GHI,LW_down,cloud_cover,precipitation,...,city,lat,lon,altitude_m,climate_zone,T_set,high_solar_resource,date,season_label,month_name
0,2024-01-01 00:00:00,0.0,21.169434,18.522064,84.866035,256.57580,0.0,359.58716,0.93454,0.000076,...,Coimbatore,11.0,77.0,411,hot-humid,45.0,1,2024-01-01,Winter,Jan
1,2024-01-01 00:00:00,0.0,8.405212,6.965942,90.658140,237.44632,0.0,272.57540,0.00000,0.000000,...,Jaisalmer,26.9,70.9,225,hot-arid,55.0,0,2024-01-01,Winter,Jan
2,2024-01-01 01:00:00,0.0,8.346008,6.958038,90.973310,239.93980,0.0,271.56085,0.00000,0.000000,...,Jaisalmer,26.9,70.9,225,hot-arid,55.0,0,2024-01-01,Winter,Jan


---
## [PIN] Map 1 — City Location & Climate Summary


In [ ]:
# ─── Colour maps ──────────────────────────────────────────────────────────────
cmap_ghi    = cm.LinearColormap(['#0d1b2a','#1b4f72','#2e86c1','#f39c12','#e74c3c','#ffffff'],
                                 vmin=df['GHI'].min(), vmax=df['GHI'].max(),
                                 caption='GHI (W/m²)')
cmap_temp   = cm.LinearColormap(['#2980b9','#82e0aa','#f9e79f','#e67e22','#c0392b'],
                                 vmin=df['T_amb'].min(), vmax=df['T_amb'].max(),
                                 caption='T_amb (°C)')
cmap_rhum   = cm.LinearColormap(['#f8c471','#abebc6','#1a9850','#145a32'],
                                 vmin=df['RHum'].min(), vmax=df['RHum'].max(),
                                 caption='RHum (%)')
cmap_cloud  = cm.LinearColormap(['#2c3e50','#7f8c8d','#bdc3c7','#ecf0f1'],
                                 vmin=0, vmax=1,
                                 caption='Cloud Cover (0–1)')
cmap_precip = cm.LinearColormap(['#fef9e7','#aed6f1','#2980b9','#1a5276'],
                                 vmin=0, vmax=df['precipitation'].quantile(0.99) if 'precipitation' in df.columns else 1,
                                 caption='Precipitation (mm)')
cmap_csi    = cm.LinearColormap(['#154360','#1a9850','#f9e79f','#e74c3c'],
                                 vmin=0, vmax=1.5,
                                 caption='CSI (Clear Sky Index)')
cmap_lw     = cm.LinearColormap(['#1c2833','#7d3c98','#e74c3c','#f39c12'],
                                 vmin=df['LW_down'].min() if 'LW_down' in df.columns else 0, 
                                 vmax=df['LW_down'].max() if 'LW_down' in df.columns else 300,
                                 caption='LW_down (W/m²)')

def season_color(month):
    if month in [12, 1, 2]: return '#2980b9'   # Winter
    elif month in [3, 4, 5]: return '#e67e22'   # Summer
    elif month in [6, 7, 8, 9]: return '#27ae60'  # Monsoon
    else: return '#8e44ad'                        # Post-monsoon

def make_popup(row, mode='hourly'):
    """Return a styled HTML popup string with safe column access."""
    def safe_val(col, default=0):
        val = getattr(row, col, default)
        try:
            return float(val) if val is not None else default
        except:
            return default
    
    if mode == 'monthly':
        html = f"""
        <div style='font-family:Arial,sans-serif;font-size:13px;width:240px;
                    background:#1e2330;color:#ecf0f1;padding:12px;border-radius:8px;'>
          <b style='font-size:15px;color:#f39c12;'>[MONTH] {getattr(row, 'month_name', 'N/A')}</b>
          <hr style='border-color:#555;margin:6px 0;'/>
          <table width='100%'>
            <tr><td>[SUN] GHI mean</td><td align='right'><b>{safe_val('GHI_mean'):.1f} W/m²</b></td></tr>
            <tr><td>[TEMP] T_amb mean</td><td align='right'><b>{safe_val('T_amb_mean'):.1f} °C</b></td></tr>
            <tr><td>[DROP] RHum mean</td><td align='right'><b>{safe_val('RHum_mean'):.1f} %</b></td></tr>
            <tr><td>[RAIN] Precip total</td><td align='right'><b>{safe_val('precip_sum'):.1f} mm</b></td></tr>
            <tr><td>[CLOUD] Cloud cover</td><td align='right'><b>{safe_val('cloud_mean'):.2f}</b></td></tr>
            <tr><td>[SKY] CSI</td><td align='right'><b>{safe_val('CSI_mean'):.3f}</b></td></tr>
            <tr><td>[COMPASS] Wind dir</td><td align='right'><b>{safe_val('W_dir_mean'):.1f}°</b></td></tr>
            <tr><td>[LIGHT] LW_down</td><td align='right'><b>{safe_val('LW_down_mean'):.1f} W/m²</b></td></tr>
          </table>
        </div>"""
    else:
        html = f"""
        <div style='font-family:Arial,sans-serif;font-size:12px;width:230px;
                    background:#1e2330;color:#ecf0f1;padding:10px;border-radius:8px;'>
          <b style='font-size:13px;color:#3498db;'>[TIME] {getattr(row, 'timestamp', 'N/A')}</b>
          <hr style='border-color:#555;margin:5px 0;'/>
          <table width='100%'>
            <tr><td>[SUN] GHI</td><td align='right'><b>{safe_val('GHI'):.1f} W/m²</b></td></tr>
            <tr><td>[TEMP] T_amb</td><td align='right'><b>{safe_val('T_amb'):.1f} °C</b></td></tr>
            <tr><td>[DROP] RHum</td><td align='right'><b>{safe_val('RHum'):.1f} %</b></td></tr>
            <tr><td>[RAIN] Precip</td><td align='right'><b>{safe_val('precipitation'):.2f} mm</b></td></tr>
            <tr><td>[CLOUD] Cloud</td><td align='right'><b>{safe_val('cloud_cover'):.2f}</b></td></tr>
            <tr><td>[SKY] CSI</td><td align='right'><b>{safe_val('CSI'):.3f}</b></td></tr>
            <tr><td>[LIGHT] LW_down</td><td align='right'><b>{safe_val('LW_down'):.1f} W/m²</b></td></tr>
            <tr><td>[WIND] W_dir</td><td align='right'><b>{safe_val('W_dir'):.1f}°</b></td></tr>
          </table>
        </div>"""
    return folium.Popup(folium.IFrame(html, width=260, height=220), max_width=280)

print('[OK] Helpers ready.')


# ─── MAP 1: City location with climate summary ─────────────────────────────

m1 = folium.Map(m1

    location=[LAT, LON],print('Map 1 — City location & annual climate summary')

    zoom_start=10,folium.LayerControl().add_to(m1)

    tiles='CartoDB positron'

)).add_to(m1)

    tooltip='Study area (±15 km from city center)'

# City center marker with full climate popup    opacity=0.3,

annual_stats_html = f"""    weight=1,

<div style='font-family:Arial,sans-serif;font-size:13px;width:280px;    fill=False,

            background:#1e2330;color:#ecf0f1;padding:12px;border-radius:8px;'>    color='red',

  <b style='font-size:16px;color:#f39c12;'>[CITY] {CITY}</b>    radius=15000,  # 15 km radius

  <hr style='border-color:#555;margin:6px 0;'/>    location=[LAT, LON],

  <table width='100%'>folium.Circle(

    <tr><td><b>Coordinates</b></td><td align='right'>{LAT:.4f}°N, {LON:.4f}°E</td></tr># Add circle to show approximate city area

    <tr><td><b>Altitude</b></td><td align='right'>{ALT} m</td></tr>

    <tr><td colspan='2'><hr style='margin:3px 0'/></td></tr>).add_to(m1)

    <tr><td>[TEMP] T_amb mean</td><td align='right'><b>{df['T_amb'].mean():.1f} °C</b></td></tr>    icon=folium.Icon(color='red', icon='home', prefix='fa')

    <tr><td>[TEMP] T_amb range</td><td align='right'>{df['T_amb'].min():.1f}–{df['T_amb'].max():.1f} °C</td></tr>    tooltip=f'{CITY} — Click for full climate summary',

    <tr><td>[SUN] GHI mean</td><td align='right'><b>{df['GHI'].mean():.1f} W/m²</b></td></tr>    popup=folium.Popup(folium.IFrame(annual_stats_html, width=310, height=320), max_width=330),

    <tr><td>[SUN] GHI peak</td><td align='right'>{df['GHI'].max():.1f} W/m²</td></tr>    location=[LAT, LON],

    <tr><td>[DROP] RHum mean</td><td align='right'><b>{df['RHum'].mean():.1f} %</b></td></tr>folium.Marker(

    <tr><td>[RAIN] Total precip</td><td align='right'>{df['precipitation'].sum():.1f} mm</td></tr>

    <tr><td>[WIND] Mean wind spd</td><td align='right'>{df['W_spd'].mean():.2f} m/s</td></tr>"""

    <tr><td colspan='2'><hr style='margin:3px 0'/></td></tr></div>

    <tr><td>Data period</td><td align='right'>{df['timestamp'].min().strftime('%Y-%m-%d')} to {df['timestamp'].max().strftime('%Y-%m-%d')}</td></tr>  </table>
    <tr><td>Sample count</td><td align='right'>{len(df)} rows</td></tr>

[OK] Helpers ready.


---
## [SUN] Map 2 — GHI Hourly Heatmap on Map
This shows the spatial marker with a **popup heatmap** of mean GHI by hour and season.


In [ ]:
# ─── MAP 2: GHI dynamic heat points (sampled daytime rows) ──────────────────
m2 = folium.Map(location=[LAT, LON], zoom_start=8,
                tiles='CartoDB dark_matter')

# Only daytime hours where GHI > 10
df_day = df[(df['hour'] >= 6) & (df['hour'] <= 18) &
            (df['GHI'] > 10)].copy() if 'GHI' in df.columns else df.copy()

# Sample up to 5000 rows for performance
df_sample = df_day.sample(min(5000, len(df_day)), random_state=42)

# Add tiny random jitter so points spread visually (±0.02°)
np.random.seed(42)
jitter = 0.02
lats = LAT + np.random.uniform(-jitter, jitter, len(df_sample))
lons = LON + np.random.uniform(-jitter, jitter, len(df_sample))
ghi_vals = df_sample['GHI'].values

# Weight = GHI normalised to [0,1]
ghi_norm = (ghi_vals - ghi_vals.min()) / (ghi_vals.max() - ghi_vals.min() + 1e-9)
heat_data = [[lats[i], lons[i], float(ghi_norm[i])] for i in range(len(df_sample))]

HeatMap(
    heat_data,
    name='GHI Heatmap',
    radius=18, blur=15, min_opacity=0.3,
    gradient={0.0: 'navy', 0.3: 'blue', 0.5: 'lime', 0.7: 'yellow', 1.0: 'red'}
).add_to(m2)

# Center marker
folium.Marker(
    [LAT, LON],
    tooltip=f'Coimbatore | Mean daytime GHI: {df_day["GHI"].mean():.1f} W/m²',
    icon=folium.DivIcon(html=f"""
        <div style='background:#e65c00;color:white;padding:4px 8px;
                    border-radius:5px;font-size:12px;font-weight:bold;
                    white-space:nowrap;border:1px solid white'>
            [SUN] GHI avg: {df_day['GHI'].mean():.0f} W/m²
        </div>""")
).add_to(m2)

# Colormap legend
colormap = cm.LinearColormap(
    colors=['navy','blue','lime','yellow','red'],
    vmin=ghi_vals.min(), vmax=ghi_vals.max(),
    caption='GHI (W/m²) — Daytime hourly samples'
)
colormap.add_to(m2)

folium.LayerControl().add_to(m2)
print('Map 2 — GHI Heatmap (daytime samples)')
m2

Map 2 — GHI Heatmap (daytime samples)


---
## [TEMP] Map 3 — Monthly Temperature & Humidity Bubble Map


In [ ]:
# ─── MAP 3: Monthly temperature bubbles ─────────────────────────────────────
m3 = folium.Map(location=[LAT, LON], zoom_start=7,
                tiles='CartoDB positron')

monthly = df.groupby('month').agg(
    T_amb_mean=('T_amb', 'mean'),
    T_amb_max=('T_amb', 'max'),
    RHum_mean=('RHum', 'mean'),
    GHI_mean=('GHI', 'mean'),
    RRTDHS_mean=('RRTDHS', 'mean') if 'RRTDHS' in df.columns else ('GHI', 'mean'),
    precip_sum=('precipitation', 'sum') if 'precipitation' in df.columns else ('GHI', 'sum'),
    month_name=('month_name', 'first')
).reset_index()

# Arrange bubbles in a circle around the city center
angles = np.linspace(0, 2 * np.pi, 12, endpoint=False)
radius_deg = 0.55   # degrees offset

season_colors = {
    1:'#1565C0', 2:'#1976D2', 3:'#EF6C00',
    4:'#F57C00', 5:'#FF8F00', 6:'#2E7D32',
    7:'#388E3C', 8:'#43A047', 9:'#4CAF50',
    10:'#6A1B9A', 11:'#7B1FA2', 12:'#0D47A1'
}

for _, row in monthly.iterrows():
    m = int(row['month'])
    angle = angles[m - 1]
    blat = LAT + radius_deg * np.cos(angle)
    blon = LON + radius_deg * np.sin(angle)

    # Radius = proportional to mean temperature (scaled)
    bub_radius = max(8, row['T_amb_mean'] * 0.8)
    color = season_colors[m]

    popup_html = f"""
    <div style='font-family:Arial;font-size:12px;width:220px'>
      <b style='color:{color};font-size:14px'>{row['month_name']}</b><br>
      <hr style='margin:3px 0'>
      [TEMP] T_amb mean: <b>{row['T_amb_mean']:.1f} °C</b><br>
      [TEMP] T_amb max:  <b>{row['T_amb_max']:.1f} °C</b><br>
      [DROP] RHum mean:  <b>{row['RHum_mean']:.1f} %</b><br>
      [SUN] GHI mean:   <b>{row['GHI_mean']:.1f} W/m²</b><br>
      [RAIN] Total rain: <b>{row['precip_sum']:.1f} mm</b>
    </div>
    """

    folium.CircleMarker(
        location=[blat, blon],
        radius=bub_radius,
        color=color, fill=True, fill_color=color, fill_opacity=0.65,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{row['month_name']}: {row['T_amb_mean']:.1f}°C | {row['RHum_mean']:.0f}% RH"
    ).add_to(m3)

    # Label
    folium.Marker(
        [blat, blon],
        icon=folium.DivIcon(
            html=f"<div style='font-size:10px;font-weight:bold;color:{color}'>{row['month_name']}</div>",
            icon_size=(30, 20), icon_anchor=(15, 10)
        )
    ).add_to(m3)

# Centre city marker
folium.Marker(
    [LAT, LON],
    icon=folium.Icon(color='red', icon='home'),
    tooltip=CITY
).add_to(m3)

print('Map 3 — Monthly temperature & humidity bubbles')
print('Bubble size ∝ mean temperature | Click for full monthly stats')
m3

Map 3 — Monthly temperature & humidity bubbles
Bubble size ∝ mean temperature | Click for full monthly stats


---
## [WIND] Map 4 — Wind Speed & Direction Scatter Map

In [8]:
# ─── MAP 4: Wind scatter ─────────────────────────────────────────────────────
m4 = folium.Map(location=[LAT, LON], zoom_start=9,
                tiles='CartoDB positron')

# Sample 400 points (daytime, wind speed > 0.5)
df_wind = df[(df['W_spd'] > 0.5)].sample(min(400, len(df)), random_state=7)

# Color by wind speed
w_max = df_wind['W_spd'].quantile(0.95)
wind_colormap = cm.LinearColormap(
    colors=['#b3e5fc', '#039be5', '#01579b', '#263238'],
    vmin=0, vmax=w_max,
    caption='Wind Speed (m/s)'
)

np.random.seed(7)
jitter_w = 0.025

for _, row in df_wind.iterrows():
    blat = LAT + np.random.uniform(-jitter_w, jitter_w)
    blon = LON + np.random.uniform(-jitter_w, jitter_w)
    wsp  = row['W_spd']
    wdir = row['W_dir'] if 'W_dir' in df.columns else 0
    color = wind_colormap(min(wsp, w_max))

    folium.CircleMarker(
        location=[blat, blon],
        radius=max(3, wsp * 1.5),
        color=color, fill=True, fill_color=color, fill_opacity=0.6,
        tooltip=f"W_spd: {wsp:.2f} m/s | W_dir: {wdir:.0f}° | "
                f"T: {row['T_amb']:.1f}°C | {row['timestamp']}"
    ).add_to(m4)

wind_colormap.add_to(m4)

# Monthly wind summary box
monthly_wind = df.groupby('month_name')['W_spd'].mean().sort_values(ascending=False)
wind_html = '<b>Monthly mean wind speed:</b><br>' + '<br>'.join(
    [f'{m}: {v:.2f} m/s' for m, v in monthly_wind.items()]
)
folium.Marker(
    [LAT + 0.6, LON - 0.5],
    icon=folium.DivIcon(
        html=f"<div style='background:white;padding:8px;border-radius:5px;"
             f"font-size:11px;border:1px solid #ccc;width:180px'>{wind_html}</div>",
        icon_size=(190, 220)
    )
).add_to(m4)

print('Map 4 — Wind scatter (circle size ∝ wind speed, color = speed)')
m4

KeyError: 'W_spd'

---
## [RRTDHS] Map 5 — Monthly Solar Resource Index

In [ ]:
# ─── MAP 5: RRTDHS monthly markers ──────────────────────────────────────────
m5 = folium.Map(location=[LAT, LON], zoom_start=7,
                tiles='CartoDB positron')

RRTDHS_THRESHOLD = 5.7

if 'RRTDHS' in df.columns:
    monthly_rr = df.groupby('month').agg(
        RRTDHS=('RRTDHS', 'mean'),
        GHI_mean=('GHI', 'mean'),
        T_amb_mean=('T_amb', 'mean'),
        month_name=('month_name', 'first')
    ).reset_index()

    angles = np.linspace(0, 2 * np.pi, 12, endpoint=False)
    radius_deg = 0.6

    rr_max = monthly_rr['RRTDHS'].max()
    rr_min = monthly_rr['RRTDHS'].min()

    for _, row in monthly_rr.iterrows():
        m  = int(row['month'])
        angle = angles[m - 1]
        blat  = LAT + radius_deg * np.cos(angle)
        blon  = LON + radius_deg * np.sin(angle)

        rr_val  = row['RRTDHS']
        is_high = rr_val > RRTDHS_THRESHOLD
        color   = '#e65c00' if is_high else '#1565C0'
        radius  = 8 + (rr_val / (rr_max + 1e-9)) * 20

        popup_html = f"""
        <div style='font-family:Arial;font-size:12px;width:210px'>
          <b style='font-size:14px;color:{color}'>{row['month_name']}</b><br>
          <hr style='margin:3px 0'>
          📐 RRTDHS: <b>{rr_val:.3f}</b><br>
          <span style='color:{color}'>
            {'🔴 HIGH solar resource (> 5.7)' if is_high else '🔵 MODERATE solar resource'}
          </span><br>
          ☀️ GHI mean: <b>{row['GHI_mean']:.1f} W/m²</b><br>
          🌡️ T_amb:    <b>{row['T_amb_mean']:.1f} °C</b>
        </div>
        """

        folium.CircleMarker(
            location=[blat, blon],
            radius=radius,
            color=color, fill=True, fill_color=color, fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=230),
            tooltip=f"{row['month_name']}: RRTDHS={rr_val:.3f} {'🔴 HIGH' if is_high else '🔵 MOD'}"
        ).add_to(m5)

        folium.Marker(
            [blat, blon],
            icon=folium.DivIcon(
                html=f"<div style='font-size:9px;font-weight:bold;color:{color};text-align:center'>"
                     f"{row['month_name']}<br>{rr_val:.2f}</div>",
                icon_size=(40, 24), icon_anchor=(20, 12)
            )
        ).add_to(m5)

    # Threshold circle
    folium.Marker(
        [LAT, LON],
        icon=folium.Icon(color='orange', icon='sun', prefix='fa'),
        tooltip=f'Coimbatore — RRTDHS threshold: {RRTDHS_THRESHOLD}'
    ).add_to(m5)

    # Legend
    legend_html = f"""<div style='position:fixed;bottom:30px;left:30px;
        background:white;padding:10px;border-radius:5px;border:1px solid #ccc;
        font-size:12px;font-family:Arial;z-index:9999'>
        <b>RRTDHS Index (Kou 2025)</b><br>
        🔴 Red  = HIGH solar resource (> {RRTDHS_THRESHOLD})<br>
        🔵 Blue = MODERATE solar resource<br>
        <i>Bubble size ∝ RRTDHS magnitude</i>
    </div>"""
    m5.get_root().html.add_child(folium.Element(legend_html))

else:
    print('RRTDHS column not found — run 02_combine.py first')

print('Map 5 — RRTDHS solar resource index per month')
print('Red = high solar resource (RRTDHS > 5.7), Blue = moderate')
m5

---
## [MAP] Map 6 — Seasonal Marker Cluster Map (CSI + GHI + T_amb)

In [ ]:
# ─── MAP 6: Season-coloured marker clusters ──────────────────────────────────
m6 = folium.Map(location=[LAT, LON], zoom_start=9,
                tiles='CartoDB positron')

season_color_map = {
    'Winter':       'blue',
    'Pre-Monsoon':  'orange',
    'Monsoon':      'green',
    'Post-Monsoon': 'purple'
}

# Only daytime, sample 150 per season
df_day6 = df[(df['hour'] >= 7) & (df['hour'] <= 17) & (df['GHI'] > 20)].copy()

for season, color in season_color_map.items():
    sub = df_day6[df_day6['season_label'] == season].sample(
        min(150, len(df_day6[df_day6['season_label'] == season])), random_state=42
    )
    if sub.empty:
        continue

    cluster = MarkerCluster(name=f'{season} ({len(sub)} pts)')

    np.random.seed(42)
    jitter_s = 0.03

    for _, row in sub.iterrows():
        blat = LAT + np.random.uniform(-jitter_s, jitter_s)
        blon = LON + np.random.uniform(-jitter_s, jitter_s)

        csi_val = row['CSI'] if 'CSI' in df.columns else 'N/A'
        popup_html = f"""
        <div style='font-family:Arial;font-size:12px'>
          <b>{season}</b> — {row['timestamp']}<br>
          ☀️ GHI: <b>{row['GHI']:.1f}</b> W/m²<br>
          🌥️ CSI: <b>{csi_val if isinstance(csi_val, str) else f'{csi_val:.3f}'}</b><br>
          🌡️ T_amb: <b>{row['T_amb']:.1f}°C</b>  |  💧 RHum: <b>{row['RHum']:.0f}%</b>
        </div>
        """
        folium.CircleMarker(
            location=[blat, blon],
            radius=5,
            color=color, fill=True, fill_color=color, fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=220),
            tooltip=f'{season} | GHI={row["GHI"]:.0f} W/m² | T={row["T_amb"]:.1f}°C'
        ).add_to(cluster)

    cluster.add_to(m6)

folium.Marker(
    [LAT, LON],
    icon=folium.Icon(color='red', icon='home'),
    tooltip=CITY
).add_to(m6)

folium.LayerControl(collapsed=False).add_to(m6)
print('Map 6 — Seasonal cluster map (click cluster to expand)')
m6

---
## [CHART] Interactive Plotly Time Series
Complementary interactive plots showing temporal trends.

In [ ]:
# ─── PLOTLY 1: GHI + T_amb dual-axis time series ────────────────────────────
daily = df.resample('D', on='timestamp').agg(
    GHI_mean=('GHI', 'mean'),
    GHI_max=('GHI', 'max'),
    T_amb_mean=('T_amb', 'mean'),
    T_amb_max=('T_amb', 'max'),
    RHum_mean=('RHum', 'mean'),
    W_spd_mean=('W_spd', 'mean'),
    cloud_cover_mean=('cloud_cover', 'mean') if 'cloud_cover' in df.columns else ('GHI', 'count'),
    precip_sum=('precipitation', 'sum') if 'precipitation' in df.columns else ('GHI', 'count'),
).reset_index()

fig1 = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        'Daily GHI — Mean & Max (W/m²)',
        'Daily Temperature (°C) & Relative Humidity (%)',
        'Daily Wind Speed (m/s) & Precipitation (mm)'
    ),
    vertical_spacing=0.07
)

# Row 1: GHI
fig1.add_trace(go.Scatter(
    x=daily['timestamp'], y=daily['GHI_mean'],
    name='GHI mean', line=dict(color='#FF8C00', width=1.5),
    fill='tozeroy', fillcolor='rgba(255,140,0,0.15)'
), row=1, col=1)
fig1.add_trace(go.Scatter(
    x=daily['timestamp'], y=daily['GHI_max'],
    name='GHI max', line=dict(color='#e65c00', width=1, dash='dot')
), row=1, col=1)

# Row 2: Temperature + RHum
fig1.add_trace(go.Scatter(
    x=daily['timestamp'], y=daily['T_amb_mean'],
    name='T_amb mean', line=dict(color='#d32f2f', width=1.5)
), row=2, col=1)
fig1.add_trace(go.Scatter(
    x=daily['timestamp'], y=daily['RHum_mean'],
    name='RHum mean', line=dict(color='#1565C0', width=1.5, dash='dash'),
    yaxis='y4'
), row=2, col=1)

# Row 3: Wind + Precip
fig1.add_trace(go.Scatter(
    x=daily['timestamp'], y=daily['W_spd_mean'],
    name='W_spd mean', line=dict(color='#00838F', width=1.5)
), row=3, col=1)
fig1.add_trace(go.Bar(
    x=daily['timestamp'], y=daily['precip_sum'],
    name='Precip (mm)', marker_color='rgba(21,101,192,0.5)'
), row=3, col=1)

fig1.update_layout(
    height=750, title_text=f'Coimbatore Climate — Daily Overview (ERA5 2024)',
    title_font_size=16, hovermode='x unified',
    legend=dict(orientation='h', y=-0.06),
    template='plotly_white'
)
fig1.show()

In [ ]:
# ─── PLOTLY 2: Monthly boxplots (GHI distribution) ───────────────────────────
fig2 = px.box(
    df[df['GHI'] > 0],
    x='month_name', y='GHI',
    color='season_label',
    category_orders={'month_name': list(MONTH_NAMES.values())},
    color_discrete_map={
        'Winter': '#1565C0',
        'Pre-Monsoon': '#EF6C00',
        'Monsoon': '#2E7D32',
        'Post-Monsoon': '#6A1B9A'
    },
    title='Monthly GHI Distribution by Season — Coimbatore (daytime only)',
    labels={'GHI': 'GHI (W/m²)', 'month_name': 'Month'},
    template='plotly_white'
)
fig2.update_layout(height=450, legend_title='Season', hovermode='x')
fig2.show()

In [ ]:
# ─── PLOTLY 3: Hourly mean profile by season ─────────────────────────────────
hourly_season = df.groupby(['season_label', 'hour']).agg(
    GHI_mean=('GHI', 'mean'),
    T_amb_mean=('T_amb', 'mean'),
    RHum_mean=('RHum', 'mean')
).reset_index()

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Mean GHI by Hour & Season', 'Mean T_amb & RHum by Hour & Season')
)

colors3 = {'Winter':'#1565C0','Pre-Monsoon':'#EF6C00','Monsoon':'#2E7D32','Post-Monsoon':'#6A1B9A'}

for season in hourly_season['season_label'].unique():
    sub = hourly_season[hourly_season['season_label'] == season]
    c = colors3.get(season, 'gray')
    fig3.add_trace(go.Scatter(
        x=sub['hour'], y=sub['GHI_mean'],
        name=season, line=dict(color=c, width=2),
        legendgroup=season
    ), row=1, col=1)
    fig3.add_trace(go.Scatter(
        x=sub['hour'], y=sub['T_amb_mean'],
        name=f'{season} T', line=dict(color=c, width=2, dash='solid'),
        legendgroup=season, showlegend=False
    ), row=1, col=2)
    fig3.add_trace(go.Scatter(
        x=sub['hour'], y=sub['RHum_mean'],
        name=f'{season} RH', line=dict(color=c, width=1.5, dash='dot'),
        legendgroup=season, showlegend=False
    ), row=1, col=2)

fig3.update_xaxes(title_text='Hour of Day', dtick=3)
fig3.update_yaxes(title_text='GHI (W/m²)', row=1, col=1)
fig3.update_yaxes(title_text='T_amb (°C) / RHum (%)', row=1, col=2)
fig3.update_layout(
    height=430, title_text='Coimbatore — Mean Hourly Profiles by Season',
    template='plotly_white', hovermode='x unified',
    legend=dict(orientation='h', y=-0.12)
)
fig3.show()

In [ ]:
# ─── PLOTLY 4: RRTDHS + CSI Monthly Bar + Line chart ─────────────────────────
monthly_kpi = df.groupby(['month', 'month_name']).agg(
    RRTDHS_mean=('RRTDHS', 'mean') if 'RRTDHS' in df.columns else ('GHI', 'mean'),
    CSI_mean=('CSI', 'mean')       if 'CSI' in df.columns    else ('GHI', 'count'),
    GHI_mean=('GHI', 'mean'),
    high_solar=('high_solar_resource', 'mean') if 'high_solar_resource' in df.columns else ('GHI', 'count')
).reset_index().sort_values('month')

bar_colors = ['#e65c00' if v > 5.7 else '#1565C0'
              for v in monthly_kpi['RRTDHS_mean']]

fig4 = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        'Monthly RRTDHS Index (🔴 > 5.7 = High solar resource)',
        'Monthly Clear Sky Index (CSI) & Mean GHI'
    ),
    vertical_spacing=0.1
)

fig4.add_trace(go.Bar(
    x=monthly_kpi['month_name'], y=monthly_kpi['RRTDHS_mean'],
    marker_color=bar_colors, name='RRTDHS'
), row=1, col=1)
fig4.add_hline(y=5.7, line_dash='dash', line_color='red',
               annotation_text='Threshold 5.7 (Kou 2025)',
               annotation_position='top right', row=1, col=1)

fig4.add_trace(go.Bar(
    x=monthly_kpi['month_name'], y=monthly_kpi['CSI_mean'],
    marker_color='#7CB9E8', name='CSI mean', opacity=0.8
), row=2, col=1)
fig4.add_trace(go.Scatter(
    x=monthly_kpi['month_name'], y=monthly_kpi['GHI_mean'],
    name='GHI mean (W/m²)', line=dict(color='#e65c00', width=2),
    yaxis='y3'
), row=2, col=1)

fig4.update_layout(
    height=600,
    title_text='Coimbatore — Monthly RRTDHS, CSI & GHI',
    template='plotly_white',
    legend=dict(orientation='h', y=-0.08)
)
fig4.show()

In [ ]:
# ─── PLOTLY 5: GHI × T_amb scatter, coloured by cloud cover ─────────────────
df_scatter = df[(df['GHI'] > 10) & (df['hour'] >= 7) & (df['hour'] <= 17)].sample(
    min(3000, len(df)), random_state=1
)

fig5 = px.scatter(
    df_scatter,
    x='T_amb', y='GHI',
    color='cloud_cover' if 'cloud_cover' in df.columns else 'CSI',
    size='RHum',
    color_continuous_scale='RdYlGn_r',
    hover_data=['timestamp', 'hour', 'CSI', 'W_spd', 'season_label'],
    title='GHI vs Ambient Temperature — coloured by Cloud Cover, sized by RHum',
    labels={'T_amb': 'Ambient Temperature (°C)', 'GHI': 'GHI (W/m²)',
            'cloud_cover': 'Cloud Cover'},
    opacity=0.55,
    template='plotly_white'
)
fig5.update_layout(height=480, coloraxis_colorbar=dict(title='Cloud Cover'))
fig5.show()

---
## [SAVE] Export All Maps as HTML Files

In [ ]:
# ─── SAVE all maps to HTML ───────────────────────────────────────────────────
import os
os.makedirs('coimbatore_maps', exist_ok=True)

maps = {
    'map1_city_summary.html':     m1,
    'map2_ghi_heatmap.html':      m2,
    'map3_monthly_temp.html':     m3,
    'map4_wind_scatter.html':     m4,
    'map5_rrtdhs_index.html':     m5,
    'map6_season_clusters.html':  m6,
}

for fname, fmap in maps.items():
    path = f'coimbatore_maps/{fname}'
    fmap.save(path)
    print(f'Saved: {path}')

# ── Optionally download from Colab ──────────────────────────────────────────
# from google.colab import files
# import shutil
# shutil.make_archive('coimbatore_maps', 'zip', 'coimbatore_maps')
# files.download('coimbatore_maps.zip')

print('\nAll maps saved to coimbatore_maps/')